# Getting Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## stdin and stdout

When you run a Python script from the command line (terminal / Command Prompt):
- stdin ("standard input") = the data coming into the script
- stdout ("standard output") = the data going out of the script

Using the pipe symbol `|`, the output of one command becomes the input of the next, like water flowing through connected pipes:

cat chai_shop_log.txt | python egrep.py "[0-9]" | python line_count.py

read the file -> keep only the lines with numbers -> count those lines.

In [2]:
from pathlib import Path

# All practice files for this chapter will go into this folder
log_text = """Monday: sold 120 cups of chai
Tea leaves delivered for the chai
Tuesday: sold 135 cups of chai
Milk was late in the morning
Wednesday: sold 110 cups of chai and 40 samosas
Shop closed early due to rain
Thursday: sold 150 cups chai
Friday: 160 chai sold
"""
data = Path("getting_data")
data.mkdir(exist_ok=True)
(data / "chai_shop_log.txt").write_text(log_text)
print(log_text)

Monday: sold 120 cups of chai
Tea leaves delivered for the chai
Tuesday: sold 135 cups of chai
Milk was late in the morning
Wednesday: sold 110 cups of chai and 40 samosas
Shop closed early due to rain
Thursday: sold 150 cups chai
Friday: 160 chai sold



In [3]:
%%writefile getting_data/egrep.py
# egrep.py: print only the lines that match a pattern (regex)
import sys, re

regex = sys.argv[1]            # sys.argv[1] = the pattern given on the command line

for line in sys.stdin:         # read every line coming IN
    if re.search(regex, line):
        sys.stdout.write(line) # send matching lines OUT

Overwriting getting_data/egrep.py


In [4]:
%%writefile getting_data/line_count.py
# line_count.py: count the lines coming in
import sys

count = 0
for line in sys.stdin:
    count += 1

print(count)  # print() also goes to stdout

Overwriting getting_data/line_count.py


In [5]:
%%writefile getting_data/most_common_words.py
# most_common_words.py: show the N most common words
import sys
from collections import Counter

try:
    num_words = int(sys.argv[1])
except (IndexError, ValueError):
    print("usage: most_common_words.py num_words")
    sys.exit(1)                # non-zero exit code = error

counter = Counter(word.lower()
                  for line in sys.stdin
                  for word in line.strip().split()
                  if word)

for word, count in counter.most_common(num_words):
    sys.stdout.write(f"{count}\t{word}\n")

Overwriting getting_data/most_common_words.py


## Reading Files

### Opening a file
- `'r'` : read only
- `'w'`: write and also deletes the old contents if the file exists! 
- `'a'`: append (add to the end)

Always use a `with` block. It closes the file automatically, even if there is an error:

```python
with open("file.txt", "r") as f:
       data = f.read()
```
here the file is already closed


In [10]:
from collections import Counter
emails_text = """priya.sharma@gmail.com
rahul@yahoo.co.in
anita@gmail.com
vikram@rediffmail.com
sneha@company.co.in
this line has no email address
arjun@GMAIL.com
meera@yahoo.co.in
ravi@mail.company.co.in
"""
(data / "email_addresses.txt").write_text(emails_text)

def get_domain(email_address):
    """Split on '@' and return the last piece (in lowercase)"""
    return email_address.lower().split("@")[-1]

with open(data / "email_addresses.txt", "r") as f:
    domain_counts = Counter(get_domain(line.strip())    # strip() removes the "\n" at the end
                            for line in f
                            if "@" in line)              # skip lines without an email

print(domain_counts.most_common())

[('gmail.com', 3), ('yahoo.co.in', 2), ('rediffmail.com', 1), ('company.co.in', 1), ('mail.company.co.in', 1)]


## Delimited Files (CSV / TSV)

Most data files have many values on each line, separated by a delimiter:
- **CSV** = Comma-Separated Values (`,`)
- **TSV** = Tab-Separated Values (`\t`)

Never split these lines yourself with `.split(",")`. Values can **contain** commas, tabs or even new lines! Use Python's `csv` module or pandas, which handle all of this properly.

### Example 1: a file with no header (tab-separated)
Daily vegetable prices at a local mandi (made-up numbers, ₹ per kg):

In [11]:
import csv
(data / "mandi_prices.txt").write_text(
    "2026-10-19\tTomato\t40\n"
    "2026-10-19\tOnion\t35\n"
    "2026-10-19\tPotato\t25\n"
    "2026-10-20\tTomato\t45\n"
    "2026-10-20\tOnion\t38\n"
    "2026-10-20\tPotato\t24\n"
)

# csv.reader gives each row as a LIST
with open(data / "mandi_prices.txt", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        date, vegetable, price = row[0], row[1], float(row[2])
        print(f"{date}  {vegetable:<7} ₹{price:.0f}/kg")

2026-10-19  Tomato  ₹40/kg
2026-10-19  Onion   ₹35/kg
2026-10-19  Potato  ₹25/kg
2026-10-20  Tomato  ₹45/kg
2026-10-20  Onion   ₹38/kg
2026-10-20  Potato  ₹24/kg


In [13]:
# With a header row, csv.DictReader gives each row as a dictionary 
# so you can use column names instead of numbers.

(data / "mandi_prices_colon.txt").write_text(
    "date:vegetable:price_per_kg\n"
    "2026-10-19:Tomato:40\n"
    "2026-10-19:Onion:35\n"
    "2026-10-19:Potato:25\n"
)

with open(data / "mandi_prices_colon.txt", newline="") as f:
    reader = csv.DictReader(f, delimiter=":")
    for row in reader:
        print(row["vegetable"], "-> ₹", float(row["price_per_kg"]))

Tomato -> ₹ 40.0
Onion -> ₹ 35.0
Potato -> ₹ 25.0
